# ДЗ 4. DML та DDL команди, складні SQL-вирази на датасеті Olist

**Датасет:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — ~100 тис. замовлень бразильського маркетплейсу (2016–2018): клієнти, замовлення, позиції, товари, продавці, відгуки. Використано 6 із 9 файлів, як вимагає завдання.

**Отримання даних:** спочатку `kagglehub.dataset_download('olistbr/brazilian-ecommerce')` (працює без облікових даних Kaggle для публічного датасету); якщо недоступно — ті самі 6 CSV лежать у репозиторії в `data/olist/` (варіант B).

**Середовище:** у Colab PostgreSQL 16 запускається через `pgserver`; локально ноутбук перевірено на Docker `postgres:16` через змінну оточення `HW4_PG_URI`. Усі DDL починаються з `DROP TABLE IF EXISTS`, тому Restart & Run All відтворюваний.

In [1]:
# ---------- Завдання 1. Setup ----------
import os, sys, subprocess, importlib

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, mod in [("psycopg2-binary", "psycopg2"), ("sqlalchemy", "sqlalchemy"), ("pandas", "pandas"), ("kagglehub", "kagglehub")]:
    ensure(pkg, mod)

import pandas as pd
from sqlalchemy import create_engine, text

PG_URI = os.environ.get("HW4_PG_URI")
if PG_URI:
    print("Зовнішній PostgreSQL:", PG_URI.split("@")[-1])
else:
    ensure("pgserver")
    import pgserver
    pg = pgserver.get_server("/tmp/hw4_pg", cleanup_mode="stop")
    PG_URI = pg.get_uri()
    print("pgserver запущено:", PG_URI.split("@")[-1])

engine = create_engine(PG_URI, future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

def q(sql, **params):
    '''SELECT -> DataFrame'''
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def run(sql):
    '''DDL/DML у транзакції. Якщо є RETURNING - повертає DataFrame.'''
    with engine.begin() as conn:
        res = conn.execute(text(sql))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)

Зовнішній PostgreSQL: localhost:55434/hw4
PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [2]:
# Отримання файлів Olist: варіант A - kagglehub, варіант B - CSV з репозиторію (data/olist/)
FILES = {
    "olist_customers_raw":     "olist_customers_dataset.csv",
    "olist_orders_raw":        "olist_orders_dataset.csv",
    "olist_order_items_raw":   "olist_order_items_dataset.csv",
    "olist_products_raw":      "olist_products_dataset.csv",
    "olist_sellers_raw":       "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

def has_all(d): return d and all(os.path.exists(os.path.join(d, f)) for f in FILES.values())

dataset_dir = None
for candidate in ["data/olist", "../data/olist", "/content/goit-rdb-hw-04/data/olist"]:
    if has_all(candidate):
        dataset_dir = candidate; print("Варіант B: CSV з репозиторію:", candidate); break
if dataset_dir is None:
    import kagglehub
    dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print("Варіант A: kagglehub:", dataset_dir)
assert has_all(dataset_dir), "Не знайдено всі 6 CSV Olist"

Варіант B: CSV з репозиторію: data/olist


In [3]:
# Raw-таблиці: точне відображення CSV, типи підбирає pandas. Це лише проміжний шар.
for table_name, file_name in FILES.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000, method="multi")
    print(f"{table_name:26s} {df.shape}")

olist_customers_raw        (99441, 5)


olist_orders_raw           (99441, 8)


olist_order_items_raw      (112650, 7)


olist_products_raw         (32951, 9)
olist_sellers_raw          (3095, 4)


olist_order_reviews_raw    (99224, 7)


### Typed-схема

Typed-таблиці — ті, з якими працюють усі наступні завдання: `TEXT`-ключі з PK/FK, `CHECK` на статус замовлення й невід'ємні розміри/ціни, `NUMERIC(12,2)` для грошей, `TIMESTAMP` для дат, identity-ключ у відгуках (бо `review_id` в Olist не унікальний). DDL взято з умови без змін структури; у `SELECT` з raw-таблиці товарів використано фактичні назви колонок CSV з помилкою `lenght`.

In [4]:
run('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id               TEXT PRIMARY KEY,
    customer_unique_id        TEXT NOT NULL,
    customer_zip_code_prefix  INTEGER,
    customer_city             TEXT,
    customer_state            CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                       TEXT PRIMARY KEY,
    customer_id                    TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                   TEXT NOT NULL,
    order_purchase_timestamp       TIMESTAMP NOT NULL,
    order_approved_at              TIMESTAMP,
    order_delivered_carrier_date   TIMESTAMP,
    order_delivered_customer_date  TIMESTAMP,
    order_estimated_delivery_date  TIMESTAMP,
    CHECK (order_status IN ('created', 'approved', 'invoiced', 'processing', 'shipped', 'delivered', 'unavailable', 'canceled'))
);

CREATE TABLE olist_products (
    product_id                  TEXT PRIMARY KEY,
    product_category_name       TEXT,
    product_name_length         INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length  INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty          INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g            INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm           INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm           INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm            INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id               TEXT PRIMARY KEY,
    seller_zip_code_prefix  INTEGER,
    seller_city             TEXT,
    seller_state            CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id             TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id        INTEGER NOT NULL,
    product_id           TEXT REFERENCES olist_products(product_id),
    seller_id            TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date  TIMESTAMP,
    price                NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value        NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id           BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id               TEXT,
    order_id                TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score            SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title    TEXT,
    review_comment_message  TEXT,
    review_creation_date    TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews (order_id);
''')
print("typed-схему створено")

typed-схему створено


In [5]:
run('''
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP, order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP, order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER, product_description_lenght::INTEGER,   -- назви з CSV, з помилкою "lenght"
       product_photos_qty::INTEGER, product_weight_g::INTEGER,
       product_length_cm::INTEGER, product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP, price::NUMERIC(12, 2), freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (review_id, order_id, review_score, review_comment_title, review_comment_message,
                                 review_creation_date, review_answer_timestamp)
SELECT review_id, order_id, review_score::SMALLINT, review_comment_title, review_comment_message,
       review_creation_date::TIMESTAMP, review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
''')
print("дані перенесено в typed-таблиці")

дані перенесено в typed-таблиці


In [6]:
q('''
-- Smoke-test: typed vs raw
SELECT 'customers'   AS table_name, (SELECT COUNT(*) FROM olist_customers_raw)     AS raw_rows, COUNT(*) AS typed_rows FROM olist_customers
UNION ALL SELECT 'orders',      (SELECT COUNT(*) FROM olist_orders_raw),        COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw),   COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products',    (SELECT COUNT(*) FROM olist_products_raw),      COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers',     (SELECT COUNT(*) FROM olist_sellers_raw),       COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews',     (SELECT COUNT(*) FROM olist_order_reviews_raw), COUNT(*) FROM olist_order_reviews
ORDER BY table_name
''')

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


**Smoke-test.** Кількість рядків у typed-таблицях збігається з raw для всіх шести таблиць (99 441 клієнтів і замовлень, 112 650 позицій, 32 951 товар, 3 095 продавців, 99 224 відгуки). Жоден рядок не загубився: усі FK (`customer_id`, `product_id`, `seller_id`, `order_id`) знайшли батьківські записи, усі статуси входять у `CHECK`, усі `review_score` в 1–5. Відгуків менше, ніж замовлень, — це не втрата при перенесенні, а властивість датасету: частина замовлень без відгуку (див. 4.5).

## Завдання 2. DML-запити з PostgreSQL-фічами

### 2.1. `INSERT ... RETURNING`: alert для продавця з найбільшою кількістю запізнень

In [7]:
run('''
DROP TABLE IF EXISTS seller_alerts CASCADE;
CREATE TABLE seller_alerts (
    alert_id    BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id   TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type  TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity    SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details     TEXT,
    created_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
''')
run('''
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT seller_id, 'late_delivery', 4,
       'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
''')

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-11-03 09:55:34.461341+00:00


**Пояснення 2.1.** Запит шукає продавця з найбільшою кількістю доставлених із запізненням замовлень (`seller_orders` з `DISTINCT`, щоб замовлення з кількома позиціями того самого продавця рахувалося один раз) і створює для нього alert. `RETURNING` повертає щойно створений рядок разом із згенерованим `alert_id` і `created_at` без окремого `SELECT` — це і є audit-log прямо в ноутбуці: видно, що створено, коли і з якими параметрами. Результат: продавець `4a3ca931…` має 172 запізнілі доставки, alert з severity 4. Запит не ідемпотентний (повторний запуск додасть другий такий самий alert), і це коректно для журналу подій; якби потрібна була унікальність, слід додати `UNIQUE (seller_id, alert_type)` і `ON CONFLICT DO NOTHING`.

### 2.2. `INSERT ... ON CONFLICT DO UPDATE`: ідемпотентний upsert `seller_score`

Пастка грануляності: `order_items ⟕ reviews` дає рівень *seller × order_item*, і один відгук на замовлення з трьома позиціями порахується тричі. Тому спочатку `seller_orders` (DISTINCT `seller_id, order_id`), окремо `review_per_order` (один рядок на замовлення), і лише потім агрегат по продавцю.

In [8]:
run('''
DROP TABLE IF EXISTS seller_score CASCADE;
CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
''')
UPSERT_SELLER_SCORE = '''
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id)                        AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT seller_id, avg_rating, n_reviews,
       CASE WHEN avg_rating IS NULL THEN NULL
            WHEN avg_rating >= 4.5 THEN 'gold'
            WHEN avg_rating >= 3.5 THEN 'silver'
            ELSE 'bronze' END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;
'''
first = run(UPSERT_SELLER_SCORE)
print("перший запуск: повернуто рядків", len(first))
first.head(8)

перший запуск: повернуто рядків 3095


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-11-03 09:55:34.539883+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-11-03 09:55:34.539883+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-11-03 09:55:34.539883+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-11-03 09:55:34.539883+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-11-03 09:55:34.539883+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-11-03 09:55:34.539883+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-11-03 09:55:34.539883+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-11-03 09:55:34.539883+00:00


In [9]:
# Повторний запуск - та сама кількість рядків у таблиці, лише updated_at змінився
second = run(UPSERT_SELLER_SCORE)
q('''
SELECT COUNT(*) AS sellers_in_score,
       COUNT(*) FILTER (WHERE avg_rating IS NULL) AS sellers_without_reviews,
       COUNT(*) FILTER (WHERE tier = 'gold')   AS gold,
       COUNT(*) FILTER (WHERE tier = 'silver') AS silver,
       COUNT(*) FILTER (WHERE tier = 'bronze') AS bronze
FROM seller_score
''')

,sellers_in_score,sellers_without_reviews,gold,silver,bronze
0,3095,5,983,1568,539


**Пояснення 2.2.** `seller_orders` використовує `SELECT DISTINCT`, бо в `order_items` один продавець може мати кілька позицій в одному замовленні; без `DISTINCT` грануляність була б seller × item, і review одного замовлення врахувався б стільки разів, скільки позицій. Відгуки спочатку агрегуються до рівня `order_id` (`review_per_order`), бо в Olist бувають кілька записів review на одне замовлення, а оцінка стосується замовлення, не продавця. `ON CONFLICT (seller_id) DO UPDATE` робить запуск повторюваним: другий виклик не додав жодного рядка (у таблиці ті самі 3 095 продавців), а лише оновив `updated_at`. Продавці без відгуків (5 із 3 095) мають `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`, бо `LEFT JOIN` залишає їх у наборі, а `AVG` по порожній множині дає NULL — це чесніше, ніж приписувати їм умовний рейтинг. Розподіл: 983 gold, 1 568 silver, 539 bronze. Спрощення датасету: review прив'язаний до замовлення, тож у замовленні з товарами кількох продавців одна оцінка зараховується кожному з них.

### 2.3. `UPDATE ... FROM`: прапорець `is_late` у `olist_orders`

In [10]:
run("ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;")
updated = run('''
WITH delivery_flags AS (
    SELECT order_id,
           (order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;
''')
print("оновлено рядків:", len(updated), "| з них is_late = TRUE:", int(updated["is_late"].sum()))
q("SELECT is_late, COUNT(*) AS n_orders, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct FROM olist_orders GROUP BY is_late ORDER BY is_late")

оновлено рядків: 99441 | з них is_late = TRUE: 6535


,is_late,n_orders,pct
0,False,92906,93.43
1,True,6535,6.57


**Пояснення 2.3.** `ALTER TABLE ... ADD COLUMN IF NOT EXISTS` робить DDL повторюваним. `UPDATE ... FROM` з CTE перераховує прапорець для всіх 99 441 замовлень за поточними датами доставки: 6 535 (6.57 %) доставлено пізніше за `order_estimated_delivery_date`. Запит ідемпотентний: при повторному запуску значення обчислюються заново, а не «дописуються», тому якщо дати в джерелі виправлять, прапорець теж виправиться. `RETURNING` повертає всі оновлені рядки — у ноутбуці це дозволяє порахувати, скільки стало `TRUE`, не роблячи окремий запит.

### 2.4. `DELETE ... RETURNING`: контрольоване видалення застарілих alert-ів

In [11]:
deleted = run('''
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
''')
print("видалено рядків:", len(deleted))
deleted

видалено рядків: 0


,alert_id,seller_id,alert_type,created_at


**Пояснення 2.4.** Видалено 0 рядків — очікувано: єдиний alert створено щойно, у цій же сесії, і умова `created_at < NOW() - INTERVAL '30 days'` для нього хибна. `DELETE ... RETURNING` корисний саме для таких контрольованих чисток: перед тим як довіритись видаленню в production, видно точний список того, що зникло, і його можна записати в archive-таблицю тією самою командою (`WITH deleted AS (DELETE ... RETURNING *) INSERT INTO seller_alerts_archive SELECT * FROM deleted`).

## Завдання 3. `CREATE TABLE customer_segments`

Сегментація рахується на рівні `customer_unique_id` (реальний покупець), а не `customer_id` (ідентифікатор замовлення/доставки). Спочатку — дедуплікований `olist_customer_dim`, на який можна повісити FK.

In [12]:
run('''
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id  TEXT PRIMARY KEY,
    latest_customer_id  TEXT REFERENCES olist_customers(customer_id),
    customer_state      CHAR(2),
    customer_city       TEXT
);

INSERT INTO olist_customer_dim (customer_unique_id, latest_customer_id, customer_state, customer_city)
SELECT DISTINCT ON (c.customer_unique_id)
       c.customer_unique_id, c.customer_id, c.customer_state, c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id;

CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
    rfm_score           SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
    recency_days        INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
    n_orders            INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE;
''')
q("SELECT COUNT(*) AS customers_raw, COUNT(DISTINCT customer_unique_id) AS unique_customers FROM olist_customers")

,customers_raw,unique_customers
0,99441,96096


In [13]:
SEGMENT_SQL = '''
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id)                     AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
           MAX(ot.order_date)                              AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
           CASE WHEN cs.monetary_value >= 1000 THEN 'VIP'
                WHEN cs.last_order_date IS NOT NULL
                     AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                     AND cs.n_orders = 1 THEN 'new'
                WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
                ELSE 'inactive' END AS segment_name,
           CASE WHEN cs.monetary_value >= 1000 THEN 5
                WHEN cs.monetary_value >= 500  THEN 4
                WHEN cs.n_orders >= 2          THEN 3
                WHEN cs.n_orders = 1           THEN 2
                ELSE 1 END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders;
'''
seg = run(SEGMENT_SQL)
print("рядків у RETURNING:", len(seg))
run(SEGMENT_SQL)   # повторний запуск - ON CONFLICT, без дублювання
q('''
SELECT segment_name, COUNT(*) AS n_customers,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct,
       ROUND(AVG(monetary_value), 2) AS avg_monetary,
       ROUND(AVG(n_orders), 3) AS avg_orders,
       ROUND(AVG(recency_days)) AS avg_recency_days
FROM customer_segments
GROUP BY segment_name
ORDER BY n_customers DESC
''')

рядків у RETURNING: 96096


,segment_name,n_customers,pct,avg_monetary,avg_orders,avg_recency_days
0,regular,45982,47.85,224.57,1.070,307.0
1,inactive,40032,41.66,59.28,1.000,316.0
2,new,8891,9.25,140.39,1.000,72.0
3,VIP,1191,1.24,1591.81,1.114,290.0


In [14]:
q('''
SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname
''')

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_custom...


**Пояснення до Завдання 3.**
- **Чому `customer_unique_id`.** У `olist_customers` 99 441 рядок, але лише 96 096 унікальних покупців: `customer_id` видається на кожне замовлення, тому RFM за ним дав би `n_orders = 1` майже для всіх. На рівні `customer_unique_id` з'являються повторні покупці (середнє `n_orders` у сегменті regular 1.07, у VIP 1.11).
- **Чому потрібна `olist_customer_dim`.** FK має посилатися на PRIMARY KEY або UNIQUE; у `olist_customers` `customer_unique_id` не унікальний, тому для посилальної цілісності створено дедупліковану вимірювальну таблицю (`DISTINCT ON` бере найновіший `customer_id` покупця). `ON DELETE CASCADE` прибере сегмент разом із покупцем.
- **Чому `reference_date` з датасету.** Olist — історичний зріз 2016–2018; якби `recency_days` рахувалися від `CURRENT_DATE`, усі покупці мали б recency у тисячі днів, сегмент `new` був би порожнім, а результат змінювався б від запуску до запуску. `MAX(order_purchase_timestamp) + 1` дає стабільну точку «наступний день після останнього замовлення».
- **Як `ON CONFLICT` робить сегментацію повторюваною.** `UNIQUE (customer_unique_id)` + `ON CONFLICT ... DO UPDATE` перетворюють повторний запуск на оновлення тих самих 96 096 рядків замість вставки дублікатів; `assigned_at = NOW()` фіксує момент перерахунку.

Результат: regular 47.9 %, inactive 41.7 %, new 9.3 %, VIP 1.2 % (середня витрата VIP — 1 592). Таблиця має identity PK, UNIQUE, FK, 5 CHECK-обмежень (перевірено через `pg_constraint`).

## Завдання 4. JOIN-запити з business-interpretation

### 4.1. INNER JOIN на чотирьох таблицях: топ категорій × штат продавця за revenue у доставлених замовленнях
**Business-question:** які категорії товарів і штати продавців дають найбільший revenue у `delivered`-замовленнях?

In [15]:
q('''
SELECT p.product_category_name, s.seller_state,
       COUNT(DISTINCT oi.order_id)                         AS n_orders,
       SUM(oi.price + oi.freight_value)::NUMERIC(12, 2)    AS revenue
FROM olist_order_items AS oi
JOIN olist_orders   AS o ON o.order_id   = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_sellers  AS s ON s.seller_id  = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10
''')

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Interpretation.** Усі 10 найприбутковіших пар «категорія × штат продавця» — зі штату SP (Сан-Паулу): продавці SP домінують настільки, що жоден інший штат не потрапляє в топ. Лідери: `cama_mesa_banho` (1.07 млн, 8 214 замовлень), `relogios_presentes` (1.02 млн при вдвічі меншій кількості замовлень — дорожчий середній чек), `beleza_saude` (0.79 млн). Чотиритабличний INNER JOIN тут коректний, бо нас цікавлять лише позиції з відомим товаром і продавцем у доставлених замовленнях; `COUNT(DISTINCT order_id)` потрібен, бо одне замовлення має кілька позицій.

### 4.2. LEFT JOIN + COALESCE: customer-level features без множення review
**Business-question:** які customer-level ознаки можна побудувати для реального покупця (`customer_unique_id`), і хто витратив найбільше?

In [16]:
q('''
WITH order_totals AS (
    SELECT o.order_id, c.customer_unique_id,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT ot.customer_unique_id,
           COUNT(DISTINCT ot.order_id)                    AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
           AVG(rpo.order_review_score)::NUMERIC(3, 2)      AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id, cd.customer_state,
       COALESCE(cs.n_orders, 0)                          AS n_orders,
       COALESCE(cs.total_spend, 0)::NUMERIC(12, 2)       AS total_spend,
       COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2)   AS avg_review_score,
       (cs.avg_review_score IS NULL)                     AS review_missing
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 10
''')

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score,review_missing
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0,False
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0,False
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0,False
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0,False
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0,True
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0,False
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0,False
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0,False
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0,False
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0,False


**Interpretation.** Топ покупців за витратами очолює клієнт із RJ з одним замовленням на 13 664 — у 60 разів більше за середню витрату regular-сегмента; 9 із 10 найбільших витрат — це один-єдиний великий чек, а не повторні покупки. `review_per_order` агрегує відгуки до рівня замовлення, тому покупець із 3 позиціями й одним відгуком отримує той самий бал один раз, а не тричі. `COALESCE(avg_review_score, 0)` заповнює відсутній відгук нулем — для ML це пастка: 0 виглядає як «найгірша оцінка», тому поруч виведено прапорець `review_missing` (у п'ятого покупця відгуку немає), і в моделі слід використовувати саме прапорець або NULL, а не 0.

### 4.3. FULL OUTER JOIN: штати серед клієнтів і продавців
**Business-question:** які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [17]:
q('''
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers FROM olist_customers GROUP BY customer_state
),
seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers FROM olist_sellers GROUP BY seller_state
)
SELECT COALESCE(c.state, s.state) AS state, c.n_customers, s.n_sellers,
       CASE WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
            WHEN c.state IS NOT NULL THEN 'customers_only'
            ELSE 'sellers_only' END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s ON s.state = c.state
ORDER BY state_presence, state
''')

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AM,148,1.0,both
2,BA,3380,19.0,both
3,CE,1336,13.0,both
4,DF,2140,30.0,both
5,ES,2033,23.0,both
6,GO,2020,40.0,both
7,MA,747,1.0,both
8,MG,11635,244.0,both
9,MS,715,5.0,both


**Interpretation.** `FULL OUTER JOIN` показує асиметрію: клієнти є в усіх 27 штатах Бразилії, продавці — лише у 23. Чотири штати (AL, AP, RR, TO) — `customers_only`: туди купують, але звідти ніхто не продає; `sellers_only` не існує. Концентрація продавців ще вища за концентрацію клієнтів: у SP 1 849 продавців із 3 095 (60 %) при 42 % клієнтів, а в AC, AM, MA, PA, PI — по одному продавцю. Для логістичної моделі це означає, що відстань «продавець → клієнт» для північних штатів майже завжди міжрегіональна.

### 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст
**Business-question:** скільки є пар продавців у межах одного штату, які можна порівнювати регіонально, і в яких штатах таких пар найбільше?

In [18]:
q('''
SELECT s1.seller_state,
       COUNT(*)                          AS n_pairs,
       COUNT(DISTINCT s1.seller_city)    AS n_cities_a,
       MIN(s1.seller_city || ' <-> ' || s2.seller_city) AS example_pair
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
  ON s1.seller_state = s2.seller_state
 AND s1.seller_id < s2.seller_id                          -- кожна пара один раз
 AND s1.seller_city IS DISTINCT FROM s2.seller_city       -- різні міста
GROUP BY s1.seller_state
ORDER BY n_pairs DESC
LIMIT 10
''')

,seller_state,n_pairs,n_cities_a,example_pair
0,SP,1457751,261,aguas claras df <-> alambari
1,PR,51665,67,almirante tamandare <-> apucarana
2,MG,27184,81,alfenas <-> andradas
3,SC,17208,64,araquari <-> ararangua
4,RJ,10151,38,04482255 <-> aperibe
5,RS,7698,51,alvorada <-> arvorezinha
6,GO,519,12,abadia de goias <-> formosa
7,ES,231,11,afonso claudio <-> cachoeiro de itapemirim
8,BA,149,12,arraial d'ajuda (porto seguro) <-> eunapolis
9,DF,57,3,brasilia <-> brasilia df


**Interpretation.** SELF JOIN із умовою `s1.seller_id < s2.seller_id` дає кожну пару один раз, а `IS DISTINCT FROM` по місту відсікає пари з одного міста. У SP таких пар 1.46 млн (261 місто), далі PR (52 тис.) і MG (27 тис.) — порівняння «всередині штату» має сенс лише для 6–7 великих штатів, у решти менше ніж 600 пар. Побічний результат — проблеми якості даних: у RJ серед назв міст трапляється `04482255` (поштовий індекс замість міста), у DF є і `brasilia`, і `brasilia df` — одне місто під двома назвами, яке self-join вважає різними. Перед регіональним аналізом поле `seller_city` потрібно нормалізувати.

### 4.5. Anti-join: замовлення без жодного відгуку
**Business-question:** скільки замовлень не мають review-запису і чи залежить це від статусу замовлення?

In [19]:
q('''
SELECT o.order_status,
       COUNT(*)                                                       AS orders_without_review,
       (SELECT COUNT(*) FROM olist_orders o2 WHERE o2.order_status = o.order_status) AS orders_total,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM olist_orders o2 WHERE o2.order_status = o.order_status), 2) AS pct_without
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_review DESC
''')

,order_status,orders_without_review,orders_total,pct_without
0,delivered,646,96478,0.67
1,shipped,75,1107,6.78
2,canceled,20,625,3.20
3,unavailable,14,609,2.30
4,processing,6,301,1.99
5,invoiced,5,314,1.59
6,created,2,5,40.00


In [20]:
q('''
-- Ті самі замовлення через NOT EXISTS (еквівалентний anti-join), перші 5 за датою
SELECT o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp
FROM olist_orders AS o
WHERE NOT EXISTS (SELECT 1 FROM olist_order_reviews rv WHERE rv.order_id = o.order_id)
ORDER BY o.order_purchase_timestamp
LIMIT 5
''')

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27


**Interpretation.** Без відгуку 768 замовлень із 99 441 (0.8 %); частка сильно залежить від статусу: серед `delivered` — лише 0.67 %, серед `shipped` — 6.8 %, серед `created` — 40 % (2 з 5). Тобто відсутність review здебільшого пов'язана з незавершеною доставкою, а не з небажанням клієнта. Обидва варіанти anti-join (`LEFT JOIN ... IS NULL` і `NOT EXISTS`) повертають той самий набір; найстаріші замовлення без відгуку — жовтень 2016, перший місяць роботи платформи.

## Завдання 5. GROUP BY, HAVING і агрегати

### 5.1. Revenue per product category, лише категорії з виручкою > 100 000

In [21]:
q('''
SELECT COALESCE(p.product_category_name, '(unknown)') AS category,
       COUNT(DISTINCT oi.order_id)        AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2)      AS gross_revenue,
       SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
       AVG(oi.price)::NUMERIC(8, 2)       AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY COALESCE(p.product_category_name, '(unknown)')
HAVING SUM(oi.price) > 100000
ORDER BY gross_revenue DESC
''')

,category,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation.** 32 категорії перевищили поріг 100 000 за `HAVING SUM(price) > 100000`. Лідер за чистою виручкою — `beleza_saude` (1.26 млн), хоча за кількістю замовлень перша `cama_mesa_banho` (9 417); `relogios_presentes` має найвищий середній чек серед масових категорій (201). Окремо стоїть `pcs`: лише 181 замовлення, але 223 тис. виручки при середній ціні 1 098. Категорія `(unknown)` — 1 451 замовлення і 180 тис. виручки без назви категорії: пропуск у `product_category_name`, який варто або відновити за описом товару, або тримати як явну категорію.

### 5.2. AOV per customer-state для штатів із ≥ 500 замовленнями (WHERE + HAVING в одному запиті)

In [22]:
q('''
WITH order_totals AS (
    SELECT o.order_id, c.customer_state,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'                    -- WHERE: фільтр РЯДКІВ до агрегації
    GROUP BY o.order_id, c.customer_state
)
SELECT customer_state,
       COUNT(*)                                            AS n_orders,
       SUM(order_total)::NUMERIC(12, 2)                    AS gmv,
       (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 500                                     -- HAVING: фільтр ГРУП після агрегації
ORDER BY aov DESC
''')

,customer_state,n_orders,gmv,aov
0,PB,517,137838.55,266.61
1,PA,946,212023.57,224.13
2,CE,1279,266436.77,208.32
3,MA,717,147803.55,206.14
4,MT,886,181224.42,204.54
5,PE,1593,308972.05,193.96
6,MS,701,134367.55,191.68
7,BA,3256,591137.81,181.55
8,GO,1957,334212.35,170.78
9,SC,3546,595127.78,167.83


**Interpretation.** Тут видно різницю `WHERE` і `HAVING` в одному запиті: `WHERE o.order_status = 'delivered'` відкидає окремі *рядки* ще до групування, а `HAVING COUNT(*) >= 500` відкидає цілі *групи* (штати) після підрахунку. Залишилось 17 штатів. Найвищий середній чек — у віддалених штатах: PB 267, PA 224, CE 208, тоді як SP з найбільшим GMV (5.77 млн, 40 501 замовлення) має найнижчий AOV 142. Пояснення — вартість доставки: `order_total` включає `freight_value`, а доставка на північ і північний схід дорожча, тому AOV там завищений не покупками, а логістикою.

### 5.3. Категорії, які купував реальний покупець (STRING_AGG), лише для покупців з ≥ 3 замовленнями

In [23]:
q('''
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id)          AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2)       AS total_spend,
       COUNT(DISTINCT COALESCE(p.product_category_name, '(unknown)')) AS n_categories,
       STRING_AGG(DISTINCT COALESCE(p.product_category_name, '(unknown)'), ', '
                  ORDER BY COALESCE(p.product_category_name, '(unknown)')) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 3
ORDER BY n_orders DESC, total_spend DESC
LIMIT 10
''')

,customer_unique_id,n_orders,total_spend,n_categories,categories
0,8d50f5eadf50201ccdcedfb9e2ac8455,16,729.62,4,"construcao_ferramentas_ferramentas, esporte_lazer, fashi..."
1,3e43e6105506432c953e165fb2acf44c,9,1000.85,5,"cama_mesa_banho, casa_construcao, informatica_acessorios..."
2,1b6c7548a2a1f9037c1fd3ddfed95f33,7,809.21,5,"beleza_saude, moveis_decoracao, papelaria, perfumaria, u..."
3,ca77025e7201e3b30c44b472ff346268,7,806.61,4,"cama_mesa_banho, cool_stuff, moveis_decoracao, utilidade..."
4,6469f99c1f9dfae7733b25662e7f1782,7,664.20,1,cama_mesa_banho
5,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,836.60,5,"automotivo, beleza_saude, cama_mesa_banho, cool_stuff, t..."
6,dc813062e0fc23409cd255f7f53c7074,6,687.76,3,"brinquedos, cool_stuff, relogios_presentes"
7,63cfc61cee11cbe306bff5857d00bfe4,6,579.81,5,"cama_mesa_banho, construcao_ferramentas_construcao, move..."
8,f0e310a6839dce9de1638e0fe5ab282a,6,438.09,3,"cama_mesa_banho, (unknown), utilidades_domesticas"
9,12f5d6e1cbf93dafd9dcc19095df0b3d,6,58.40,3,"fashion_bolsas_e_acessorios, moveis_quarto, pet_shop"


**Interpretation.** `STRING_AGG(DISTINCT ...)` збирає категорії покупця в один рядок; `HAVING COUNT(DISTINCT order_id) >= 3` лишає лише справді повторних покупців. Їх небагато: лідер зробив 16 замовлень у 4 категоріях на 730; другий — 9 замовлень на 1 001 у 5 категоріях. Повторні покупці купують у різних категоріях (3–5), тобто лояльність до платформи, а не до товару, — крім одного покупця з 7 замовленнями лише в `cama_mesa_banho`. Для рекомендаційної моделі це сигнал, що cross-category ознаки будуть інформативнішими за «улюблену категорію».

## Завдання 6. Set operations, window function, Reflection

### 6.1. EXCEPT: штати, де є клієнти, але немає жодного продавця

In [24]:
q('''
SELECT customer_state AS state FROM olist_customers
EXCEPT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state
''')

,state
0,AL
1,AP
2,RR
3,TO


In [25]:
q('''
-- Для контрасту: UNION (без дублікатів) vs UNION ALL (з дублікатами) на тих самих наборах
SELECT 'UNION'     AS op, COUNT(*) AS n FROM (SELECT customer_state AS s FROM olist_customers UNION     SELECT seller_state FROM olist_sellers) t
UNION ALL
SELECT 'INTERSECT',       COUNT(*)      FROM (SELECT customer_state AS s FROM olist_customers INTERSECT SELECT seller_state FROM olist_sellers) t
UNION ALL
SELECT 'EXCEPT',          COUNT(*)      FROM (SELECT customer_state AS s FROM olist_customers EXCEPT    SELECT seller_state FROM olist_sellers) t
''')

,op,n
0,UNION,27
1,INTERSECT,23
2,EXCEPT,4


**Пояснення 6.1.** Обрано `EXCEPT`, бо питання «де є клієнти, але немає продавців» — це буквально різниця множин. Результат: AL, AP, RR, TO — ті самі чотири штати, що в 4.3 позначені `customers_only`, тобто `FULL OUTER JOIN` і `EXCEPT` дають узгоджену відповідь різними засобами. Контрольний запит: `UNION` — 27 штатів (усі, без дублікатів), `INTERSECT` — 23 (і клієнти, і продавці), `EXCEPT` — 4; 23 + 4 = 27, як і має бути. `UNION ALL` тут не підійшов би, бо він зберігає дублікати рядків і повернув би 99 441 + 3 095 рядків замість списку штатів.

### 6.2. Window function: найновіше замовлення кожного покупця з ≥ 2 замовленнями (ROW_NUMBER) і ранг штатів за GMV (RANK)

In [26]:
q('''
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp, o.order_status,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                              ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC) AS rn,
           COUNT(*)     OVER (PARTITION BY c.customer_unique_id) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
)
SELECT customer_unique_id, n_orders, order_id, order_purchase_timestamp AS latest_order_ts, order_status
FROM ranked_orders
WHERE rn = 1 AND n_orders >= 2
ORDER BY n_orders DESC, latest_order_ts DESC
LIMIT 10
''')

,customer_unique_id,n_orders,order_id,latest_order_ts,order_status
0,8d50f5eadf50201ccdcedfb9e2ac8455,17,d3582fd5ccccd9cb229a63dfb417c86f,2018-08-20 19:14:26,delivered
1,3e43e6105506432c953e165fb2acf44c,9,ff89ef7b3952bba5ac06d61c4a79ffbe,2018-02-27 18:36:39,delivered
2,6469f99c1f9dfae7733b25662e7f1782,7,ccb1046449f1a494de800567e7cb5979,2018-06-28 00:43:34,delivered
3,ca77025e7201e3b30c44b472ff346268,7,8dacd1da7c4dc40d5c59e7ddd93b3ac8,2018-06-01 11:38:29,delivered
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7,43f08913407cac4e74a898d968e58c1a,2018-02-14 13:22:12,delivered
5,dc813062e0fc23409cd255f7f53c7074,6,f0031384d36ff045ee1e866942d0fa7a,2018-08-23 00:07:26,delivered
6,63cfc61cee11cbe306bff5857d00bfe4,6,c29713133e5e33937af1a6e1e1370552,2018-05-28 17:20:02,delivered
7,f0e310a6839dce9de1638e0fe5ab282a,6,f2270ae03c6578b9cb6e4bba286bc962,2018-04-05 09:04:45,delivered
8,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,b86ec88f5bffbf1f61fb5daed332e355,2018-01-24 15:15:26,delivered
9,de34b16117594161a6a89c50b289d35a,6,d29eca3633b0b60adf268cae9133cd93,2018-01-17 23:34:21,delivered


In [27]:
q('''
WITH state_gmv AS (
    SELECT c.customer_state, SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS gmv
    FROM olist_order_items AS oi
    JOIN olist_orders AS o ON o.order_id = oi.order_id
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
)
SELECT customer_state, gmv,
       RANK()       OVER (ORDER BY gmv DESC)                                  AS gmv_rank,
       ROUND(gmv * 100.0 / SUM(gmv) OVER (), 2)                              AS pct_of_total,
       ROUND(SUM(gmv) OVER (ORDER BY gmv DESC) * 100.0 / SUM(gmv) OVER (), 2) AS cumulative_pct
FROM state_gmv
ORDER BY gmv_rank
LIMIT 10
''')

,customer_state,gmv,gmv_rank,pct_of_total,cumulative_pct
0,SP,5921678.12,1,37.38,37.38
1,RJ,2129681.98,2,13.44,50.82
2,MG,1856161.49,3,11.72,62.53
3,RS,885826.76,4,5.59,68.12
4,PR,800935.44,5,5.06,73.18
5,BA,611506.67,6,3.86,77.04
6,SC,610213.60,7,3.85,80.89
7,DF,353229.44,8,2.23,83.12
8,GO,347706.93,9,2.19,85.32
9,ES,324801.91,10,2.05,87.37


**Пояснення 6.2.** `ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY ... DESC)` нумерує замовлення кожного покупця від найновішого; `WHERE rn = 1` лишає останнє, а віконний `COUNT(*)` без `ORDER BY` додає загальну кількість замовлень тим самим проходом, без GROUP BY і без втрати деталей рядка. Найактивніший покупець має 17 замовлень (у 5.3 було 16, бо там рахувались лише замовлення з позиціями), останнє — 20 серпня 2018. Другий запит з `RANK()` і кумулятивною сумою показує концентрацію ринку: SP дає 37.4 % GMV, три штати (SP, RJ, MG) — 62.5 %, десять — 87.4 %. Такий «Парето-профіль» через вікно обчислюється одним запитом замість підзапитів із самоз'єднанням.

## Reflection

**Найскладніший JOIN** — FULL OUTER JOIN: не синтаксично, а тому, що результат треба читати з обох боків одночасно і правильно обробляти NULL через `COALESCE`, інакше штати `customers_only` губляться. Self join теж вимагав дисципліни: без `s1.seller_id < s2.seller_id` кожна пара дублювалась, а в SP це мільйони зайвих рядків.

**Найважче як business-question** формулювався self join: «пари продавців з різних міст одного штату» — технічно просто, але довелося пояснити, кому і навіщо такий список потрібен (регіональні бенчмарки), і одразу стало видно, що без нормалізації міст він некоректний.

**Найкорисніша PostgreSQL-фіча для production-ETL** — `INSERT ... ON CONFLICT DO UPDATE`: вона робить крок пайплайна ідемпотентним, а в парі з `RETURNING` дає журнал змін без окремих запитів. Identity-ключі та constraints — фундамент, але саме upsert перетворює перерахунок у безпечну повторювану операцію.

**Window functions у ML-pipeline** я б використав для лагових ознак покупця (попереднє замовлення, дні від попередньої покупки через `LAG`), ковзних сум витрат за 90 днів, рангу продавця всередині категорії та для дедуплікації «остання версія запису» через `ROW_NUMBER`.

**Data-quality проблеми Olist:** 768 замовлень без відгуку, `review_id`, що не унікальний, 1 451 замовлення з товарами без категорії, міста на кшталт `04482255` і `brasilia df`, 5 продавців без жодного відгуку, статус `created` у 5 замовлень, майже порожні тексти відгуків.

**`customer_unique_id`, а не `customer_id`**, бо другий видається на кожне замовлення: 99 441 проти 96 096, і RFM за ним не бачить повторних покупок.

**Грануляність seller × order** перед `seller_score` потрібна, щоб один відгук на замовлення з кількома позиціями того самого продавця не рахувався кілька разів і не викривляв середній рейтинг.